In [1]:
import os
import pandas as pd

# Step 1: Define the folder path
folder_path = r"D:\master_reserch\data\algo_wise\cluster\user_info"

# Step 2: Get all CSV files in the folder
csv_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]

# Step 3: Process each file
columns_to_remove = ['discrimination', 'difficulty', 'distance_to_line']
cleaned_dfs = []

for file in csv_files:
    algo_name = file.split("_")[-1].replace(".csv", "")  # e.g., "CKE", "KGIN"
    file_path = os.path.join(folder_path, file)

    df = pd.read_csv(file_path)

    # Drop unwanted columns if they exist
    df = df.drop(columns=[col for col in columns_to_remove if col in df.columns], errors='ignore')

    # Add algorithm presence indicator
    df[algo_name] = 1
    cleaned_dfs.append(df)

# Step 4: Combine all data
combined_df = pd.concat(cleaned_dfs, axis=0)

# Step 5: Aggregate by user_id
algo_columns = [file.split("_")[-1].replace(".csv", "") for file in csv_files]
aggregated_df = combined_df.groupby("user_id").agg({
    **{col: "first" for col in combined_df.columns if col not in algo_columns + ["user_id"]},
    **{algo: "max" for algo in algo_columns}
}).reset_index()

# Step 6: Fill NaNs in algo flags and count how many files each user appears in
for algo in algo_columns:
    aggregated_df[algo] = aggregated_df[algo].fillna(0).astype(int)

aggregated_df["files_present_in"] = aggregated_df[algo_columns].sum(axis=1)

# Step 7: Save result
output_path = os.path.join(folder_path, "merged_user_data.csv")
aggregated_df.to_csv(output_path, index=False)

print(f"✅ Cleaned and merged data saved to:\n{output_path}")


✅ Cleaned and merged data saved to:
D:\master_reserch\data\algo_wise\cluster\user_info\merged_user_data.csv
